**Part 1**

Testing class on data. First, import my .py file to define my class.

In [131]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from functools import reduce
import pyspark.sql.types as T
import pandas as pd

In [51]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config('spark.sql.ansi.enabled', 'false').getOrCreate()

In [132]:
import Project2_Part1_WarrenE

In [133]:
import importlib
importlib.reload(Project2_Part1_WarrenE)

<module 'Project2_Part1_WarrenE' from '/home/jupyter-eawarre7@ncsu.edu/Project2_Part1_WarrenE.py'>

In [134]:
csv_data = 'air.csv'
my_instance = Project2_Part1_WarrenE. \
        SparkDataCheck.from_csv(spark, csv_data)
my_instance.df = my_instance.df \
        .withColumn("over100", \
        F.when(my_instance.df["NMHC(GT)"] > 100, "over 100") \
        .otherwise("less/equal to 100")) \
        .withColumn("cold", \
        F.when(my_instance.df["T"] < 13, "cold") \
        .otherwise("not cold")) \
        .replace(-200, None) \
        .drop("_c0") \
        .withColumnsRenamed({"CO(GT)": "CO", \
                             "PT08.S1(CO)": "S1", \
                             "NMHC(GT)": "NMHC", \
                             "C6H6(GT)": "C6H6", \
                             "PT08.S2(NMHC)": "S2", \
                             "NOx(GT)": "NOX", \
                             "PT08.S3(NOx)": "S3", \
                             "NO2(GT)": "NO2", \
                             "PT08.S4(NO2)": "S4", \
                             "PT08.S5(O3)": "S5"})

In [135]:
my_instance.df.columns

['Date',
 'Time',
 'CO',
 'S1',
 'NMHC',
 'C6H6',
 'S2',
 'NOX',
 'S3',
 'NO2',
 'S4',
 'S5',
 'T',
 'RH',
 'AH',
 'over100',
 'cold']

In [136]:
my_instance.df.printSchema()

root
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- CO: double (nullable = true)
 |-- S1: integer (nullable = true)
 |-- NMHC: integer (nullable = true)
 |-- C6H6: double (nullable = true)
 |-- S2: integer (nullable = true)
 |-- NOX: integer (nullable = true)
 |-- S3: integer (nullable = true)
 |-- NO2: integer (nullable = true)
 |-- S4: integer (nullable = true)
 |-- S5: integer (nullable = true)
 |-- T: double (nullable = true)
 |-- RH: double (nullable = true)
 |-- AH: double (nullable = true)
 |-- over100: string (nullable = false)
 |-- cold: string (nullable = false)



In [137]:
my_instance.check_limits("CO", None, None).df.show(3)

No upper or lower bounds provided
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+
only showing top 3 rows


In [138]:
my_instance.check_limits("CO", None, 10).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
only showing top 3 rows


In [139]:
my_instance.check_limits("CO", 1.5, None).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
only showing top 3 rows


In [140]:
my_instance.check_limits("CO", 1.5, 4).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+
only showing top 3 rows


In [141]:
my_instance.check_string("over100", ["over 100"]).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
only showing 

In [142]:
my_instance.check_string("over100", ["over 100", "less/equal to 100"]).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
only showing 

In [143]:
my_instance.check_string("CO", ["over 100"]).df.show(3)

Supplied column is not string type. No changes have been made to the dataframe.
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|    true|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|    true|
+---------+-------------------+---+----+----+----+----+---+----+---+----+-

In [146]:
my_instance.check_string("cold", ["hot"]).df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+
only showing 

In [147]:
my_instance.check_missing("CO").df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|isMissing|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|    false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------

In [148]:
my_instance.check_missing("T").df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|isMissing|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|    false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------

In [149]:
my_instance.check_missing("RH").df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|isMissing|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|    false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------

In [150]:
my_instance.check_missing("Date").df.show(3)

+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|     Date|               Time| CO|  S1|NMHC|C6H6|  S2|NOX|  S3|NO2|  S4|  S5|   T|  RH|    AH|          over100|    cold|inBounds|inLevels|isMissing|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------+-----------------+--------+--------+--------+---------+
|3/10/2004|2026-03-29 18:00:00|2.6|1360| 150|11.9|1046|166|1056|113|1692|1268|13.6|48.9|0.7578|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 19:00:00|2.0|1292| 112| 9.4| 955|103|1174| 92|1559| 972|13.3|47.7|0.7255|         over 100|not cold|    true|   false|    false|
|3/10/2004|2026-03-29 20:00:00|2.2|1402|  88| 9.0| 939|131|1140|114|1555|1074|11.9|54.0|0.7502|less/equal to 100|    cold|    true|   false|    false|
+---------+-------------------+---+----+----+----+----+---+----+---+----+----+----+----+------

In [151]:
my_instance.get_minmax("RH")

,min(RH),max(RH)
0,9.2,88.7


In [152]:
my_instance.get_minmax("RH", "over100")

,over100,min(RH),max(RH)
0,less/equal to 100,9.2,88.7
1,over 100,14.9,81.8


In [153]:
my_instance.get_minmax("Date")

Error: The supplied column is not numeric type


In [154]:
my_instance.get_minmax(None)

,min(CO),max(CO),min(S1),max(S1),min(NMHC),max(NMHC),min(C6H6),max(C6H6),min(S2),max(S2),...,min(S4),max(S4),min(S5),max(S5),min(T),max(T),min(RH),max(RH),min(AH),max(AH)
0,0.1,11.9,647,2040,7,1189,0.1,63.7,383,2214,...,551,2775,221,2523,-1.9,44.6,9.2,88.7,0.1847,2.231


In [156]:
my_instance.get_minmax(None, "cold")

,cold,min(CO),max(CO),cold,min(S1),max(S1),cold,min(NMHC),max(NMHC),cold,...,max(S5),cold,min(T),max(T),cold,min(RH),max(RH),cold,min(AH),max(AH)
0,not cold,0.1,10.2,not cold,667,2040,not cold,9,1189,not cold,...,2519,not cold,13.0,44.6,not cold,9.2,87.2,not cold,0.2185,2.2310
1,cold,0.1,11.9,cold,647,2008,cold,7,1042,cold,...,2523,cold,-1.9,12.9,cold,13.5,88.7,cold,0.1847,1.2422


In [158]:
my_instance.get_stringlevels("over100")

+-----------------+-----+
|          over100|count|
+-----------------+-----+
|less/equal to 100| 8787|
|         over 100|  570|
+-----------------+-----+



In [159]:
my_instance.get_stringlevels("cold")

+--------+-----+
|    cold|count|
+--------+-----+
|not cold| 6317|
|    cold| 3040|
+--------+-----+



In [160]:
my_instance.get_stringlevels("over100", "cold")

+-----------------+--------+-----+
|          over100|    cold|count|
+-----------------+--------+-----+
|less/equal to 100|    cold| 2912|
|less/equal to 100|not cold| 5875|
|         over 100|not cold|  442|
|         over 100|    cold|  128|
+-----------------+--------+-----+



In [161]:
my_instance.get_stringlevels("over100", "CO")

In [162]:
my_instance.get_stringlevels("CO", "cold")

Error: All supplied columns are not StringType. Please supply a string column.


In [165]:
# Get a regular pandas dataframe from our csv
pdf = pd.read_csv("air.csv")

# Create an instance of our class from the pandas dataframe
pd_instance = Project2_Part1_WarrenE.SparkDataCheck.from_pandas(spark, pdf)

# Clean our instance dataframe as above
pd_instance.df = pd_instance.df \
        .withColumn("over100", \
        F.when(pd_instance.df["NMHC(GT)"] > 100, "over 100") \
        .otherwise("less/equal to 100")) \
        .withColumn("cold", \
        F.when(pd_instance.df["T"] < 13, "cold") \
        .otherwise("not cold")) \
        .replace(-200, None) \
        .drop("_c0") \
        .withColumnsRenamed({"CO(GT)": "CO", \
                             "PT08.S1(CO)": "S1", \
                             "NMHC(GT)": "NMHC", \
                             "C6H6(GT)": "C6H6", \
                             "PT08.S2(NMHC)": "S2", \
                             "NOx(GT)": "NOX", \
                             "PT08.S3(NOx)": "S3", \
                             "NO2(GT)": "NO2", \
                             "PT08.S4(NO2)": "S4", \
                             "PT08.S5(O3)": "S5"})

In [167]:
pd_instance.get_minmax("NMHC")

,min(NMHC),max(NMHC)
0,7,1189
